# Online Linear Autoencoder with Continual Learning| Strategy | Buffer Size | Replay Weight | Replay Batch ||----------|------------|---------------|-------------|| **Naive** (no CL) | - | - | - || **ER Scaled** | 50,000 | 0.7 | 10,000 || **ER Aggressive** | 100,000 | 1.0 | 20,000 |**Architecture**: Linear AE with windowed input (15 timesteps x 4 vars = 60-dim)\**Training**: 20 temporal windows x 100 epochs per window

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
"""
Online Linear Autoencoder with Continual Learning Strategies
"""

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch.utils.data import Dataset
import pyarrow.csv as pv
from torcheval.metrics import PeakSignalNoiseRatio
from torchmetrics.image import StructuralSimilarityIndexMeasure
from matplotlib.gridspec import GridSpec
import matplotlib.pyplot as plt
import time
import os
import json


# Dataset: Windowed AE
class WindowedAEDataset(Dataset):
    """
    Per-spatial-point dataset with temporal windowing for online AE training.
    Each window: (num_points, time_seq * num_vars) with global normalization.
    """
    def __init__(self, filepath, num_windows=20):
        print("Loading dataset from {}".format(filepath))
        read_options = pv.ReadOptions(
            column_names=['x', 'y', 'z', 't', 'Vx', 'Vy', 'Pressure', 'TKE']
        )
        table = pv.read_csv(filepath, read_options=read_options)
        data = table.to_pandas()
        data = data.sort_values(['x', 'y', 'z', 't']).reset_index(drop=True)

        fields = data[['Vx', 'Vy', 'Pressure', 'TKE']].values.astype(np.float32)
        coords = data[['x', 'y', 'z']].values.astype(np.float32)

        self.num_timesteps = data['t'].nunique()
        self.num_points = len(data) // self.num_timesteps
        self.num_vars = 4
        self.var_names = ['Vx', 'Vy', 'Pressure', 'TKE']
        self.num_windows = num_windows
        self.time_seq = self.num_timesteps // num_windows

        self.fields_3d = fields.reshape(self.num_points, self.num_timesteps, self.num_vars)
        self.field_min = fields.min(axis=0)
        self.field_max = fields.max(axis=0)
        self.field_range = self.field_max - self.field_min
        self.field_range[self.field_range == 0] = 1.0
        self.fields_3d_norm = (self.fields_3d - self.field_min) / self.field_range
        self.coords = coords.reshape(self.num_points, self.num_timesteps, 3)[:, 0, :]
        self.window_input_dim = self.time_seq * self.num_vars
        self.unique_times = np.sort(data['t'].unique())

        print("Dataset: {} points x {} timesteps, {} windows of {} timesteps".format(
            self.num_points, self.num_timesteps, self.num_windows, self.time_seq))
        print("Window input dim: {}".format(self.window_input_dim))

    def get_window_data(self, window_idx):
        start_t = window_idx * self.time_seq
        end_t = start_t + self.time_seq
        if window_idx == self.num_windows - 1:
            end_t = self.num_timesteps
        window_fields = self.fields_3d_norm[:, start_t:end_t, :]
        return torch.FloatTensor(window_fields.reshape(self.num_points, -1))

    def get_normalization_params(self):
        return {
            'field_min': self.field_min.tolist(),
            'field_max': self.field_max.tolist(),
            'field_range': self.field_range.tolist(),
            'num_timesteps': self.num_timesteps,
            'num_points': self.num_points,
            'num_vars': self.num_vars,
            'num_windows': self.num_windows,
            'time_seq': self.time_seq,
            'window_input_dim': self.window_input_dim,
        }

    def __len__(self):
        return self.num_points


# Models
class LinearEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dims, latent_dim, dropout=0.1):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h_dim in hidden_dims:
            layers.extend([nn.Linear(prev_dim, h_dim), nn.LeakyReLU(0.1), nn.Dropout(dropout)])
            prev_dim = h_dim
        layers.append(nn.Linear(prev_dim, latent_dim))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


class LinearDecoder(nn.Module):
    def __init__(self, latent_dim, hidden_dims, output_dim, dropout=0.1):
        super().__init__()
        layers = []
        prev_dim = latent_dim
        for h_dim in hidden_dims:
            layers.extend([nn.Linear(prev_dim, h_dim), nn.LeakyReLU(0.1), nn.Dropout(dropout)])
            prev_dim = h_dim
        layers.append(nn.Linear(prev_dim, output_dim))
        self.network = nn.Sequential(*layers)

    def forward(self, z):
        return self.network(z)


class LinearAutoEncoder(nn.Module):
    def __init__(self, input_dim, encoder_hidden, decoder_hidden, latent_dim, dropout=0.1):
        super().__init__()
        self.encoder = LinearEncoder(input_dim, encoder_hidden, latent_dim, dropout)
        self.decoder = LinearDecoder(latent_dim, decoder_hidden, input_dim, dropout)
        self.latent_dim = latent_dim
        self.input_dim = input_dim

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        return self.decoder(z)


class AEForwardWrapper(nn.Module):
    """Wraps AE so forward() returns only reconstruction (strategy compatibility)."""
    def __init__(self, ae):
        super().__init__()
        self.ae = ae
    def forward(self, x):
        x_hat, _ = self.ae(x)
        return x_hat


# Model Configs (for 60-dim windowed input)
AE_CONFIGS = {
    'base': {
        'encoder_hidden': [64, 32],
        'decoder_hidden': [32, 64],
        'latent_dim': 8,
        'dropout': 0.1,
    },
    'medium': {
        'encoder_hidden': [128, 64],
        'decoder_hidden': [64, 128],
        'latent_dim': 16,
        'dropout': 0.1,
    },
    'large': {
        'encoder_hidden': [256, 128, 64],
        'decoder_hidden': [64, 128, 256],
        'latent_dim': 32,
        'dropout': 0.1,
    },
}


def create_ae(size, input_dim):
    cfg = AE_CONFIGS[size]
    return LinearAutoEncoder(input_dim=input_dim, **cfg)


# CL Strategies
class ReplayBuffer:
    def __init__(self, max_size=10000):
        self.max_size = max_size
        self.inputs = None
        self.targets = None
        self.count = 0
        self.current_size = 0

    def add_window_batch(self, inputs, targets, n_samples=None):
        window_size = inputs.shape[0]
        if n_samples is None:
            n_samples = min(window_size, self.max_size // 2)
        indices = torch.randperm(window_size)[:n_samples]
        self.add_window(inputs[indices], targets[indices])

    def add_window(self, inputs, targets):
        inputs_cpu = inputs.detach().cpu()
        targets_cpu = targets.detach().cpu()
        n = inputs_cpu.shape[0]
        if self.inputs is None:
            n_init = min(n, self.max_size)
            self.inputs = inputs_cpu[:n_init].clone()
            self.targets = targets_cpu[:n_init].clone()
            self.current_size = n_init
            self.count = n_init
            start = n_init
        else:
            start = 0
        for i in range(start, n):
            self.count += 1
            if self.current_size < self.max_size:
                self.inputs = torch.cat([self.inputs, inputs_cpu[i:i+1]], dim=0)
                self.targets = torch.cat([self.targets, targets_cpu[i:i+1]], dim=0)
                self.current_size += 1
            else:
                j = np.random.randint(0, self.count)
                if j < self.max_size:
                    self.inputs[j] = inputs_cpu[i]
                    self.targets[j] = targets_cpu[i]

    def sample(self, batch_size, device=None):
        actual = min(batch_size, self.current_size)
        idx = torch.randperm(self.current_size)[:actual]
        bi, bt = self.inputs[idx], self.targets[idx]
        if device:
            bi, bt = bi.to(device), bt.to(device)
        return bi, bt

    def __len__(self):
        return self.current_size


class NaiveStrategy:
    def __init__(self):
        self.name = "naive"
    def before_window(self, model, window_idx, window_inputs, window_targets, device):
        pass
    def compute_loss(self, model, criterion, outputs, targets, window_inputs, device):
        return criterion(outputs, targets)
    def after_window(self, model, window_idx, window_inputs, window_targets, device):
        pass
    def get_config(self):
        return {"strategy": self.name}


class ERStrategy:
    def __init__(self, name, buffer_size, replay_weight, replay_batch_size):
        self.name = name
        self.buffer = ReplayBuffer(max_size=buffer_size)
        self.replay_weight = replay_weight
        self.replay_batch_size = replay_batch_size

    def before_window(self, model, window_idx, window_inputs, window_targets, device):
        pass

    def compute_loss(self, model, criterion, outputs, targets, window_inputs, device):
        current_loss = criterion(outputs, targets)
        if len(self.buffer) == 0:
            return current_loss
        replay_inputs, replay_targets = self.buffer.sample(self.replay_batch_size, device=device)
        replay_outputs = model(replay_inputs)
        replay_loss = criterion(replay_outputs, replay_targets)
        return current_loss + self.replay_weight * replay_loss

    def after_window(self, model, window_idx, window_inputs, window_targets, device):
        self.buffer.add_window_batch(window_inputs, window_targets)

    def get_config(self):
        return {"strategy": self.name, "buffer_size": self.buffer.max_size,
                "replay_weight": self.replay_weight, "replay_batch_size": self.replay_batch_size}


# Metrics
def compute_ae_psnr_ssim(predictions, targets, device):
    predictions, targets = predictions.to(device), targets.to(device)
    psnr_metric = PeakSignalNoiseRatio().to(device)
    psnr_metric.update(predictions, targets)
    psnr = psnr_metric.compute().item()
    pred_ssim = predictions.unsqueeze(0).unsqueeze(0)
    target_ssim = targets.unsqueeze(0).unsqueeze(0)
    ssim_metric = StructuralSimilarityIndexMeasure(gaussian_kernel=False, kernel_size=1).to(device)
    ssim_metric.update(pred_ssim, target_ssim)
    ssim = ssim_metric.compute().item()
    return psnr, ssim


def compute_ae_relative_error(predictions, targets):
    return (torch.norm(predictions - targets) / torch.norm(targets) * 100).item()


# Training Loop
def train_online_ae_cl(model, dataset, device, epochs_per_window, model_name,
                       output_dir, strategy, num_windows=20):
    os.makedirs(output_dir, exist_ok=True)
    wrapped = AEForwardWrapper(model).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    metrics = {'window': [], 'loss': [], 'psnr': [], 'ssim': [],
               'relative_error': [], 'time_per_window': []}
    total_params = sum(p.numel() for p in model.parameters())

    print("\n[Training] {} | {} | {:,} params | latent={}".format(
        model_name, strategy.name, total_params, model.latent_dim))

    total_start = time.time()
    for window_idx in range(num_windows):
        window_start = time.time()
        window_data = dataset.get_window_data(window_idx).to(device)
        strategy.before_window(wrapped, window_idx, window_data, window_data, device)

        wrapped.train()
        for epoch in range(epochs_per_window):
            optimizer.zero_grad()
            outputs = wrapped(window_data)
            loss = strategy.compute_loss(wrapped, criterion, outputs,
                                          targets=window_data, window_inputs=window_data, device=device)
            loss.backward()
            optimizer.step()

        strategy.after_window(wrapped, window_idx, window_data, window_data, device)

        wrapped.eval()
        with torch.no_grad():
            preds = wrapped(window_data)
        loss_val = criterion(preds, window_data).item()
        psnr, ssim = compute_ae_psnr_ssim(preds, window_data, device)
        rel_error = compute_ae_relative_error(preds, window_data)
        wt = time.time() - window_start

        metrics['window'].append(window_idx + 1)
        metrics['loss'].append(loss_val)
        metrics['psnr'].append(psnr)
        metrics['ssim'].append(ssim)
        metrics['relative_error'].append(rel_error)
        metrics['time_per_window'].append(wt)

        print("  Window {:2d}/{}: PSNR={:.2f} dB, SSIM={:.4f}, RE={:.2f}%, Time={:.2f}s".format(
            window_idx + 1, num_windows, psnr, ssim, rel_error, wt))

    total_time = time.time() - total_start
    torch.save(model.state_dict(), os.path.join(output_dir, '{}_final.pth'.format(model_name)))
    with open(os.path.join(output_dir, '{}_normalization.json'.format(model_name)), 'w') as f:
        json.dump(dataset.get_normalization_params(), f, indent=2)

    print("[Done] {:.2f}s | Final: PSNR={:.2f}, SSIM={:.4f}, RE={:.2f}%".format(
        total_time, metrics['psnr'][-1], metrics['ssim'][-1], metrics['relative_error'][-1]))

    return metrics


def evaluate_full_dataset_ae(model, dataset, device, model_name=None):
    wrapped = AEForwardWrapper(model).to(device)
    wrapped.eval()
    all_preds, all_targets = [], []
    for w in range(dataset.num_windows):
        wd = dataset.get_window_data(w).to(device)
        with torch.no_grad():
            all_preds.append(wrapped(wd))
        all_targets.append(wd)
    all_preds = torch.cat(all_preds, dim=0)
    all_targets = torch.cat(all_targets, dim=0)

    psnr, ssim = compute_ae_psnr_ssim(all_preds, all_targets, device)
    rel_error = compute_ae_relative_error(all_preds, all_targets)
    training_time = 0  # filled later

    label = model_name or "AE"
    print("[Full Eval] {} | PSNR={:.2f} dB, SSIM={:.4f}, RE={:.2f}%".format(
        label, psnr, ssim, rel_error))
    return {'psnr_db': psnr, 'ssim': ssim, 'relative_error_pct': rel_error, 'training_time_s': 0}


# Flow Field Visualization for AE
def visualize_ae_flow_field(model, dataset, device, timestep_idx=0,
                            title='AE Flow Field', save_path=None):
    """
    Visualize AE reconstruction for a specific timestep.
    Finds which window the timestep belongs to, runs the model,
    and extracts the timestep's predictions.
    """
    wrapped = AEForwardWrapper(model).to(device)
    wrapped.eval()

    # Determine window and position within window
    window_idx = min(timestep_idx // dataset.time_seq, dataset.num_windows - 1)
    local_idx = timestep_idx - window_idx * dataset.time_seq

    window_data = dataset.get_window_data(window_idx).to(device)
    with torch.no_grad():
        preds_flat = wrapped(window_data).cpu().numpy()
    targets_flat = window_data.cpu().numpy()

    # Reshape: (num_points, time_seq * num_vars) -> (num_points, time_seq, num_vars)
    preds_3d = preds_flat.reshape(dataset.num_points, dataset.time_seq, dataset.num_vars)
    targets_3d = targets_flat.reshape(dataset.num_points, dataset.time_seq, dataset.num_vars)

    preds_t = np.clip(preds_3d[:, local_idx, :], 0, 1)
    targets_t = targets_3d[:, local_idx, :]
    abs_err = np.abs(targets_t - preds_t)

    x_phys = dataset.coords[:, 0]
    y_phys = dataset.coords[:, 1]
    field_names = dataset.var_names

    fig = plt.figure(figsize=(20, 20))
    gs = GridSpec(4, 5, figure=fig, width_ratios=[1, 1, 0.05, 1, 0.05],
                  wspace=0.35, hspace=0.25)

    for row in range(4):
        original = targets_t[:, row]
        predicted = preds_t[:, row]
        error = abs_err[:, row]

        ax0 = fig.add_subplot(gs[row, 0])
        ax0.scatter(x_phys, y_phys, c=original, cmap='jet', s=0.5, alpha=0.8, vmin=0, vmax=1)
        ax0.set_title('Original: {}'.format(field_names[row]))
        ax0.set_aspect('equal'); ax0.grid(True, alpha=0.3)

        ax1 = fig.add_subplot(gs[row, 1])
        sc2 = ax1.scatter(x_phys, y_phys, c=predicted, cmap='jet', s=0.5, alpha=0.8, vmin=0, vmax=1)
        ax1.set_title('Predicted: {}'.format(field_names[row]))
        ax1.set_aspect('equal'); ax1.grid(True, alpha=0.3)

        cax1 = fig.add_subplot(gs[row, 2])
        fig.colorbar(sc2, cax=cax1)

        ax2 = fig.add_subplot(gs[row, 3])
        sc3 = ax2.scatter(x_phys, y_phys, c=error, cmap='hot', s=0.5, alpha=0.8, vmin=0, vmax=1)
        ax2.set_title('Abs Error: {}'.format(field_names[row]))
        ax2.set_aspect('equal'); ax2.grid(True, alpha=0.3)

        cax2 = fig.add_subplot(gs[row, 4])
        fig.colorbar(sc3, cax=cax2)

    fig.suptitle(title, fontsize=14, fontweight='bold', y=0.98)
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

    t_val = dataset.unique_times[timestep_idx]
    print("  Timestep: {:.4f} (index {}, window {}, local {})".format(
        t_val, timestep_idx, window_idx + 1, local_idx))


print("All utilities loaded.")

In [ ]:
DATA_FILE = "/kaggle/input/ml-test-loader-original-data-csv/ML_test_loader_original_data.csv"
EPOCHS_PER_WINDOW = 100
NUM_WINDOWS = 20
VIS_TIMESTEP = 150  # middle timestep for flow field visualization

STRATEGIES = {
    'naive': lambda: NaiveStrategy(),
    'er_scaled': lambda: ERStrategy('er_scaled', 50000, 0.7, 10000),
    'er_aggressive': lambda: ERStrategy('er_aggressive', 100000, 1.0, 20000),
}

AE_OFFLINE_REF = {
    'base':   {'psnr': 36.23, 'ssim': 0.9697, 're': 2.74},
    'medium': {'psnr': 37.90, 'ssim': 0.9719, 're': 2.26},
    'large':  {'psnr': 37.94, 'ssim': 0.9749, 're': 2.25},
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device: {}".format(device))

dataset = WindowedAEDataset(DATA_FILE, num_windows=NUM_WINDOWS)

all_metrics = {}
all_evals = {}

---## 1. Naive Online (No CL)

In [ ]:
for model_name in ['base', 'medium', 'large']:
    out_dir = "/kaggle/working/results/ae_naive_{}/".format(model_name)
    model = create_ae(model_name, dataset.window_input_dim).to(device)
    strategy = STRATEGIES['naive']()
    metrics = train_online_ae_cl(
        model=model, dataset=dataset, device=device,
        epochs_per_window=EPOCHS_PER_WINDOW, model_name=model_name,
        output_dir=out_dir, strategy=strategy, num_windows=NUM_WINDOWS)
    eval_res = evaluate_full_dataset_ae(model, dataset, device, model_name=model_name)
    eval_res['training_time_s'] = sum(metrics['time_per_window'])
    all_metrics[(model_name, 'naive')] = metrics
    all_evals[(model_name, 'naive')] = eval_res

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for mn, col in [('base', 'tab:blue'), ('medium', 'tab:orange'), ('large', 'tab:green')]:
    m = all_metrics[(mn, 'naive')]
    axes[0,0].plot(m['window'], m['loss'], '-o', color=col, lw=2, ms=4, label=mn.capitalize())
    axes[0,1].plot(m['window'], m['psnr'], '-o', color=col, lw=2, ms=4, label=mn.capitalize())
    axes[1,0].plot(m['window'], m['ssim'], '-o', color=col, lw=2, ms=4, label=mn.capitalize())
    axes[1,1].plot(m['window'], m['relative_error'], '-o', color=col, lw=2, ms=4, label=mn.capitalize())
for ax in axes.flat: ax.legend(); ax.grid(True, alpha=0.3)
axes[0,0].set(xlabel='Window', ylabel='Loss', title='Training Loss')
axes[0,1].set(xlabel='Window', ylabel='PSNR (dB)', title='PSNR')
axes[1,0].set(xlabel='Window', ylabel='SSIM', title='SSIM')
axes[1,1].set(xlabel='Window', ylabel='RE (%)', title='Relative Error')
plt.suptitle('Online AE: Naive (No CL): Per-Window Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/results/ae_naive_curves.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Naive: Flow Field Visualization (best model)
best_mn = max(['base', 'medium', 'large'], key=lambda m: all_evals[(m, 'naive')]['psnr_db'])
best_psnr = all_evals[(best_mn, 'naive')]['psnr_db']
best_model = create_ae(best_mn, dataset.window_input_dim).to(device)
best_model.load_state_dict(torch.load(
    '/kaggle/working/results/ae_naive_{}/{}_final.pth'.format(best_mn, best_mn), map_location=device))
print('Best Naive model: {} (PSNR: {:.2f} dB)'.format(best_mn, best_psnr))
visualize_ae_flow_field(
    best_model, dataset, device, timestep_idx=VIS_TIMESTEP,
    title='Online AE Naive ({}): PSNR: {:.2f} dB'.format(best_mn, best_psnr),
    save_path='/kaggle/working/results/ae_naive_flow_field.png')

---## 2. ER Scaled

In [ ]:
for model_name in ['base', 'medium', 'large']:
    out_dir = "/kaggle/working/results/ae_er_scaled_{}/".format(model_name)
    model = create_ae(model_name, dataset.window_input_dim).to(device)
    strategy = STRATEGIES['er_scaled']()
    metrics = train_online_ae_cl(
        model=model, dataset=dataset, device=device,
        epochs_per_window=EPOCHS_PER_WINDOW, model_name=model_name,
        output_dir=out_dir, strategy=strategy, num_windows=NUM_WINDOWS)
    eval_res = evaluate_full_dataset_ae(model, dataset, device, model_name=model_name)
    eval_res['training_time_s'] = sum(metrics['time_per_window'])
    all_metrics[(model_name, 'er_scaled')] = metrics
    all_evals[(model_name, 'er_scaled')] = eval_res

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for mn, col in [('base', 'tab:blue'), ('medium', 'tab:orange'), ('large', 'tab:green')]:
    m = all_metrics[(mn, 'er_scaled')]
    axes[0,0].plot(m['window'], m['loss'], '-o', color=col, lw=2, ms=4, label=mn.capitalize())
    axes[0,1].plot(m['window'], m['psnr'], '-o', color=col, lw=2, ms=4, label=mn.capitalize())
    axes[1,0].plot(m['window'], m['ssim'], '-o', color=col, lw=2, ms=4, label=mn.capitalize())
    axes[1,1].plot(m['window'], m['relative_error'], '-o', color=col, lw=2, ms=4, label=mn.capitalize())
for ax in axes.flat: ax.legend(); ax.grid(True, alpha=0.3)
axes[0,0].set(xlabel='Window', ylabel='Loss', title='Training Loss')
axes[0,1].set(xlabel='Window', ylabel='PSNR (dB)', title='PSNR')
axes[1,0].set(xlabel='Window', ylabel='SSIM', title='SSIM')
axes[1,1].set(xlabel='Window', ylabel='RE (%)', title='Relative Error')
plt.suptitle('Online AE: ER Scaled: Per-Window Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/results/ae_er_scaled_curves.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ER Scaled: Flow Field Visualization (best model)
best_mn = max(['base', 'medium', 'large'], key=lambda m: all_evals[(m, 'er_scaled')]['psnr_db'])
best_psnr = all_evals[(best_mn, 'er_scaled')]['psnr_db']
best_model = create_ae(best_mn, dataset.window_input_dim).to(device)
best_model.load_state_dict(torch.load(
    '/kaggle/working/results/ae_er_scaled_{}/{}_final.pth'.format(best_mn, best_mn), map_location=device))
print('Best ER Scaled model: {} (PSNR: {:.2f} dB)'.format(best_mn, best_psnr))
visualize_ae_flow_field(
    best_model, dataset, device, timestep_idx=VIS_TIMESTEP,
    title='Online AE ER Scaled ({}): PSNR: {:.2f} dB'.format(best_mn, best_psnr),
    save_path='/kaggle/working/results/ae_er_scaled_flow_field.png')

---## 3. ER Aggressive

In [ ]:
for model_name in ['base', 'medium', 'large']:
    out_dir = "/kaggle/working/results/ae_er_aggressive_{}/".format(model_name)
    model = create_ae(model_name, dataset.window_input_dim).to(device)
    strategy = STRATEGIES['er_aggressive']()
    metrics = train_online_ae_cl(
        model=model, dataset=dataset, device=device,
        epochs_per_window=EPOCHS_PER_WINDOW, model_name=model_name,
        output_dir=out_dir, strategy=strategy, num_windows=NUM_WINDOWS)
    eval_res = evaluate_full_dataset_ae(model, dataset, device, model_name=model_name)
    eval_res['training_time_s'] = sum(metrics['time_per_window'])
    all_metrics[(model_name, 'er_aggressive')] = metrics
    all_evals[(model_name, 'er_aggressive')] = eval_res

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for mn, col in [('base', 'tab:blue'), ('medium', 'tab:orange'), ('large', 'tab:green')]:
    m = all_metrics[(mn, 'er_aggressive')]
    axes[0,0].plot(m['window'], m['loss'], '-o', color=col, lw=2, ms=4, label=mn.capitalize())
    axes[0,1].plot(m['window'], m['psnr'], '-o', color=col, lw=2, ms=4, label=mn.capitalize())
    axes[1,0].plot(m['window'], m['ssim'], '-o', color=col, lw=2, ms=4, label=mn.capitalize())
    axes[1,1].plot(m['window'], m['relative_error'], '-o', color=col, lw=2, ms=4, label=mn.capitalize())
for ax in axes.flat: ax.legend(); ax.grid(True, alpha=0.3)
axes[0,0].set(xlabel='Window', ylabel='Loss', title='Training Loss')
axes[0,1].set(xlabel='Window', ylabel='PSNR (dB)', title='PSNR')
axes[1,0].set(xlabel='Window', ylabel='SSIM', title='SSIM')
axes[1,1].set(xlabel='Window', ylabel='RE (%)', title='Relative Error')
plt.suptitle('Online AE: ER Aggressive: Per-Window Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/results/ae_er_aggressive_curves.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ER Aggressive: Flow Field Visualization (best model)
best_mn = max(['base', 'medium', 'large'], key=lambda m: all_evals[(m, 'er_aggressive')]['psnr_db'])
best_psnr = all_evals[(best_mn, 'er_aggressive')]['psnr_db']
best_model = create_ae(best_mn, dataset.window_input_dim).to(device)
best_model.load_state_dict(torch.load(
    '/kaggle/working/results/ae_er_aggressive_{}/{}_final.pth'.format(best_mn, best_mn), map_location=device))
print('Best ER Aggressive model: {} (PSNR: {:.2f} dB)'.format(best_mn, best_psnr))
visualize_ae_flow_field(
    best_model, dataset, device, timestep_idx=VIS_TIMESTEP,
    title='Online AE ER Aggressive ({}): PSNR: {:.2f} dB'.format(best_mn, best_psnr),
    save_path='/kaggle/working/results/ae_er_aggressive_flow_field.png')

---## 4. Cross-Strategy Comparison

In [ ]:
COMP_DIR = "/kaggle/working/results/comparison_ae_online"
os.makedirs(COMP_DIR, exist_ok=True)

print("=" * 90)
print("  FULL-DATASET EVALUATION: ONLINE LINEAR AE")
print("=" * 90)
print("{:<12s} {:<16s} {:>10s} {:>10s} {:>10s} {:>10s}".format("Model", "Strategy", "PSNR", "SSIM", "RE(%)", "Time(s)"))
print("-" * 90)
for mn in ['base', 'medium', 'large']:
    r = AE_OFFLINE_REF[mn]
    print("{:<12s} {:<16s} {:>10.2f} {:>10.4f} {:>10.2f} {:>10s}".format(mn, "Offline(ref)", r["psnr"], r["ssim"], r["re"], "-"))
    for sn in ['naive', 'er_scaled', 'er_aggressive']:
        if (mn, sn) in all_evals:
            ev = all_evals[(mn, sn)]
            print("{:<12s} {:<16s} {:>10.2f} {:>10.4f} {:>10.2f} {:>10.1f}".format(
                mn, sn, ev["psnr_db"], ev["ssim"], ev["relative_error_pct"], ev["training_time_s"]))
    print("")
print("=" * 90)

In [ ]:
# Per-window PSNR: best model per strategy
fig, ax = plt.subplots(figsize=(14, 7))
sm = {'naive': ('Naive', '#F44336', '--'), 'er_scaled': ('ER Scaled', '#FF9800', '-'),
      'er_aggressive': ('ER Aggressive', '#7B1FA2', '-')}
for sn, (lb, co, ls) in sm.items():
    bm = max(['base', 'medium', 'large'], key=lambda m: all_evals[(m, sn)]['psnr_db'])
    m = all_metrics[(bm, sn)]
    ax.plot(m['window'], m['psnr'], ls, color=co, lw=2.5, marker='o', ms=5,
            label='{} ({})'.format(lb, bm))
ax.set(xlabel='Temporal Window', ylabel='PSNR (dB)',
       title='Online AE: Per-Window PSNR: Strategy Comparison')
ax.legend(fontsize=11); ax.grid(True, alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig(os.path.join(COMP_DIR, 'ae_online_psnr_per_window.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Gap to offline: grouped bar chart
fig, ax = plt.subplots(figsize=(14, 7))
x_pos = np.arange(3); width = 0.25
for i, (sn, (lb, co, _)) in enumerate(sm.items()):
    gaps = [AE_OFFLINE_REF[mn]['psnr'] - all_evals[(mn, sn)]['psnr_db'] for mn in ['base', 'medium', 'large']]
    bars = ax.bar(x_pos + i*width, gaps, width, label=lb, color=co, edgecolor='black', lw=0.5)
    for b, v in zip(bars, gaps):
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.2, '{:.1f}'.format(v),
                ha='center', fontsize=9)
ax.set_xticks(x_pos + width); ax.set_xticklabels(['Base', 'Medium', 'Large'])
ax.set(ylabel='PSNR Gap to Offline (dB)', title='Online AE: Gap to Offline (lower is better)')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(os.path.join(COMP_DIR, 'ae_online_gap_to_offline.png'), dpi=300, bbox_inches='tight')
plt.show()

---## 5. Save and Download

In [ ]:
combined = {}
for (mn, sn), ev in all_evals.items():
    combined["{}_{}".format(mn, sn)] = ev
with open(os.path.join(COMP_DIR, 'ae_online_all_results.json'), 'w') as f:
    json.dump(combined, f, indent=2)
print("Results saved. {} experiments.".format(len(combined)))

# Save per-strategy metrics CSVs
for (mn, sn), m in all_metrics.items():
    df = pd.DataFrame(m)
    csv_path = "/kaggle/working/results/ae_{}_{}/{}_metrics.csv".format(sn, mn, mn)
    if os.path.exists(os.path.dirname(csv_path)):
        df.to_csv(csv_path, index=False)

import shutil
zip_path = shutil.make_archive('/kaggle/working/ae_online_results', 'zip', '/kaggle/working/results')
print("Archive: {} ({:.1f} MB)".format(zip_path, os.path.getsize(zip_path)/(1024*1024)))